# Bin to TXT

Convert one eDAS phase `.bin` file into a `.txt` file.

- Output unit: `rad`
- Output format: 5 decimal places
- Output layout: each column is one spatial point
- Compare one selected point before and after conversion
- Diagnose which points are all-zero and which points contain signal


In [11]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from phase_bin_tools import (
    read_single_channel_phase_bin,
    read_single_channel_phase_bin_raw,
)

%matplotlib qt

def convert_bin_to_txt(bin_path, txt_path=None, points_per_frame=None, delimiter="\t"):
    """Convert one phase bin file to a txt file in radians."""
    bin_path = Path(bin_path)
    if not bin_path.exists():
        raise FileNotFoundError(f"File not found: {bin_path}")

    if txt_path is None:
        txt_path = bin_path.with_suffix(".txt")
    else:
        txt_path = Path(txt_path)

    phase_data_rad = read_single_channel_phase_bin(
        bin_path,
        points_per_frame=points_per_frame,
    )

    np.savetxt(
        txt_path,
        phase_data_rad,
        fmt="%.5f",
        delimiter=delimiter,
    )

    print(f"Input : {bin_path}")
    print(f"Output: {txt_path}")
    print(f"TXT matrix shape: {phase_data_rad.shape} -> rows=frames, cols=points")
    print("Unit  : rad")

    return txt_path, phase_data_rad


def inspect_point_before_after(bin_path, txt_path, point_index, points_per_frame=None, delimiter="\t"):
    """Show one point signal before conversion and after conversion."""
    raw_data = read_single_channel_phase_bin_raw(
        bin_path,
        points_per_frame=points_per_frame,
    )
    txt_data = np.loadtxt(txt_path, delimiter=delimiter)

    txt_data = np.atleast_2d(txt_data)
    if raw_data.shape != txt_data.shape:
        raise ValueError(
            f"Shape mismatch: raw_data.shape={raw_data.shape}, txt_data.shape={txt_data.shape}"
        )
    if not 0 <= point_index < raw_data.shape[1]:
        raise IndexError(f"point_index {point_index} out of range [0, {raw_data.shape[1] - 1}]")

    raw_signal = raw_data[:, point_index]
    txt_signal = txt_data[:, point_index]

    print(f"Selected point index: {point_index}")
    print(f"TXT matrix shape   : {txt_data.shape}")
    print("First 10 samples before conversion (raw int32):")
    print(raw_signal[:10])
    print("First 10 samples after conversion (rad, from txt):")
    print(txt_signal[:10])

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(raw_signal, linewidth=1.0)
    axes[0].set_title(f"Point {point_index} before conversion (raw int32)")
    axes[0].set_ylabel("Raw Value")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(txt_signal, linewidth=1.0, color="tab:orange")
    axes[1].set_title(f"Point {point_index} after conversion (rad, from txt)")
    axes[1].set_xlabel("Frame Index")
    axes[1].set_ylabel("Phase (rad)")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    return raw_signal, txt_signal, txt_data.shape


def diagnose_points(bin_path, points_per_frame=None, preview_count=10):
    """Diagnose which point columns are all-zero and which contain signal."""
    raw_data = read_single_channel_phase_bin_raw(
        bin_path,
        points_per_frame=points_per_frame,
    )

    col_min = raw_data.min(axis=0)
    col_max = raw_data.max(axis=0)
    nonzero_mask = np.any(raw_data != 0, axis=0)
    nonzero_points = np.flatnonzero(nonzero_mask)
    zero_points = np.flatnonzero(~nonzero_mask)

    print(f"Raw matrix shape: {raw_data.shape}")
    print(f"All-zero point count: {zero_points.size}")
    print(f"Non-zero point count: {nonzero_points.size}")
    print(f"First {preview_count} all-zero points: {zero_points[:preview_count]}")
    print(f"First {preview_count} non-zero points: {nonzero_points[:preview_count]}")

    if nonzero_points.size > 0:
        first_point = int(nonzero_points[0])
        print(f"Suggested point_index with signal: {first_point}")
        print(
            f"Point {first_point} raw min/max: "
            f"{int(col_min[first_point])} / {int(col_max[first_point])}"
        )
        print("First 10 samples at suggested point:")
        print(raw_data[:10, first_point])
    else:
        print("No non-zero points found. This bin file may itself be all zero.")

    return zero_points, nonzero_points, col_min, col_max


In [ ]:
# Update these parameters before running.
bin_file = Path(r"E:\codes\eDASread\singledataread\0000003-fs-eDAS-2000Hz-0042pt-20260421T124046.100.bin")
point_index = 10
points_per_frame = None
delimiter = "\t"

# If the file name already contains `XXXXpt`, you can keep points_per_frame=None.
# Otherwise, set it explicitly, for example: points_per_frame=819
txt_file, phase_data_rad = convert_bin_to_txt(
    bin_file,
    txt_path=None,
    points_per_frame=points_per_frame,
    delimiter=delimiter,
)


Input : E:\codes\eDASread\singledataread\0000003-fs-eDAS-2000Hz-0042pt-20260421T124046.100.bin
Output: E:\codes\eDASread\singledataread\0000003-fs-eDAS-2000Hz-0042pt-20260421T124046.100.txt
TXT matrix shape: (8000, 42) -> rows=frames, cols=points
Unit  : rad


In [9]:
# Diagnose whether point_index=0 is naturally all-zero,
# and find point columns that actually contain signal.
zero_points, nonzero_points, col_min, col_max = diagnose_points(
    bin_file,
    points_per_frame=points_per_frame,
)


Raw matrix shape: (8000, 42)
All-zero point count: 1
Non-zero point count: 41
First 10 all-zero points: [0]
First 10 non-zero points: [ 1  2  3  4  5  6  7  8  9 10]
Suggested point_index with signal: 1
Point 1 raw min/max: -7230677 / -6702753
First 10 samples at suggested point:
[-6703210 -6703066 -6704101 -6704137 -6702753 -6703064 -6703011 -6704101
 -6704533 -6704784]


In [12]:
# Compare the selected point signal before and after conversion,
# and output the txt matrix size.
raw_signal, txt_signal, txt_shape = inspect_point_before_after(
    bin_file,
    txt_file,
    point_index=point_index,
    points_per_frame=points_per_frame,
    delimiter=delimiter,
)

print(f"Output txt matrix size: {txt_shape}")


Selected point index: 10
TXT matrix shape   : (8000, 42)
First 10 samples before conversion (raw int32):
[-300081 -299902 -299800 -299970 -300376 -299974 -299936 -300043 -299873
 -300033]
First 10 samples after conversion (rad, from txt):
[-28.77078 -28.75362 -28.74384 -28.76014 -28.79907 -28.76052 -28.75688
 -28.76714 -28.75084 -28.76618]
Output txt matrix size: (8000, 42)


In [ ]:
# Preview the first 5 rows and first 5 columns.
phase_data_rad[:5, :5]
